In [6]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080122

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


[Conda 环境] base
[学号] 20244080122
[机器指纹] b1eeb302b80bf458
[RUN_ID] 20244080122-b1eeb302b80bf458


In [7]:
# 1. 回文判断：忽略空格和大小写
def is_palindrome(s):
    s_clean = s.replace(" ", "").lower()
    return s_clean == s_clean[::-1]

# 2. 字母顺序判断：忽略空格
def is_abecedarian(s):
    letters = s.replace(" ", "").lower()
    for i in range(1, len(letters)):
        if letters[i] < letters[i - 1]:
            return False
    return True

# 3. 去掉所有元音
def remove_vowels(s):
    vowels = "aeiouAEIOU"
    return "".join(ch for ch in s if ch not in vowels)


# 测试
print(is_palindrome("tacocat"), is_palindrome("TACO cat"), is_palindrome("hello"))
print(is_abecedarian("adder"), is_abecedarian("beet"), is_abecedarian("dog"), is_abecedarian("abcd efgh xyz"))
print(remove_vowels("cat"), remove_vowels("audio"), remove_vowels("aeiou"))


True True False
True True False True
ct d 


In [8]:
# 1. 列表反转，非列表引发 TypeError
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("输入必须是列表")
    return lst[::-1]

# 2. 判断是否升序，只遍历一次
def is_sorted(seq):
    for i in range(1, len(seq)):
        if seq[i] < seq[i - 1]:
            return False
    return True


# 测试
print(list_reverse([1, 2, 3]))
print(is_sorted([1, 2, 3, 4]), is_sorted([1, 3, 2]), is_sorted([]), is_sorted([5]))


[3, 2, 1]
True False True True


In [9]:
# 1. 向量数乘
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError("第一个参数必须是元组")
    if not isinstance(s, (int, float)):
        raise TypeError("标量必须是整数或浮点数")
    for x in t:
        if not isinstance(x, (int, float)):
            raise TypeError("向量元素必须是整数或浮点数")
    return tuple(x * s for x in t)

# 2. 向量内积
def vec_inner_product(x, y):
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("两个参数都必须是元组")
    if len(x) != len(y):
        raise ValueError("两个向量长度必须一致")
    return float(sum(a * b for a, b in zip(x, y)))

# 3. 检查矩阵是否有效
def check_valid_mx(mx):
    if not isinstance(mx, tuple) or len(mx) == 0:
        return False
    for row in mx:
        if not isinstance(row, tuple):
            return False
    n = len(mx[0])
    for row in mx:
        if len(row) != n:
            return False
        for x in row:
            if not isinstance(x, (int, float)):
                return False
    return True

# 4. 矩阵乘向量
def mx_vec_mult(mx, v):
    if not check_valid_mx(mx):
        raise TypeError("矩阵不合法")
    if not isinstance(v, tuple):
        raise TypeError("向量必须是元组")
    if len(mx[0]) != len(v):
        raise ValueError("矩阵列数与向量长度不一致")
    return tuple(float(sum(a * b for a, b in zip(row, v))) for row in mx)


# 测试
print(vec_scalar_mult((1, 2, 3), 2))
print(vec_inner_product((1, 2, 3), (4, 5, 6)))
print(check_valid_mx(((1, 2), (3, 4))), check_valid_mx(((1, 2), (3, 4, 5))))
print(mx_vec_mult(((1, 2), (3, 4)), (1, 1)))


(2, 4, 6)
32.0
True False
(3.0, 7.0)


**维度和元素应作为实例属性。** 理由：每个向量的维度和元素都各不相同，属于单个对象的状态；类属性是所有对象共享的，若设为类属性会被其他向量覆盖。


In [10]:
class Vector:
    def __init__(self, dim, elements=None):
        if not isinstance(dim, int) or dim < 0:
            raise TypeError("维度必须是非负整数")
        if elements is None:
            elements = tuple(0 for _ in range(dim))
        if not isinstance(elements, (list, tuple)):
            raise TypeError("元素必须是列表或元组")
        if len(elements) != dim:
            raise ValueError("维度与元素个数不一致")
        for x in elements:
            if not isinstance(x, (int, float)):
                raise TypeError("元素必须是整数或浮点数")
        self.dim = dim
        self.elements = tuple(elements)

    def _check(self, other):
        if not isinstance(other, Vector):
            raise TypeError("参数必须是 Vector 对象")
        if other.dim != self.dim:
            raise ValueError("两个向量维度必须一致")

    def __eq__(self, other):
        self._check(other)
        return self.elements == other.elements

    def __lt__(self, other):
        self._check(other)
        return self.elements < other.elements

    def __le__(self, other):
        self._check(other)
        return self.elements <= other.elements

    def __gt__(self, other):
        self._check(other)
        return self.elements > other.elements

    def __ge__(self, other):
        self._check(other)
        return self.elements >= other.elements

    def dot(self, other):
        self._check(other)
        return sum(a * b for a, b in zip(self.elements, other.elements))

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Vector(self.dim, tuple(x * other for x in self.elements))
        if isinstance(other, Vector):
            self._check(other)
            return Vector(self.dim, tuple(a * b for a, b in zip(self.elements, other.elements)))
        raise TypeError("不支持的乘法类型")

    def __rmul__(self, other):
        return self.__mul__(other)

    def __repr__(self):
        return f"Vector({self.dim}, {self.elements})"


# 测试
v1 = Vector(2, (2, 4))
v2 = Vector(2, (2, 5))
v3 = Vector(2, (2, 4))
print(v1 == v2, v1 == v3, v1 < v2, v1 <= v3, v2 > v1)
print("内积:", v1.dot(v2))
print(2 * v1, v1 * 2)
print(v1 * v3)


False True True True True
内积: 24
Vector(2, (4, 8)) Vector(2, (4, 8))
Vector(2, (4, 16))
